#**The Boston Housing Dataset**


**O Conjunto de Dados Habitacionais de Boston**

O Conjunto de Dados Habitacionais de Boston é derivado de informações coletadas pelo Serviço de Censo dos EUA sobre habitação na área de Boston, Massachusetts. A seguir, são descritas as colunas do conjunto de dados:

*   **CRIM** - taxa de criminalidade per capita por cidade
*   **ZN** - proporção de terrenos residenciais zoneados para lotes com mais de 25.000 pés quadrados
*   **INDUS** - proporção de acres de negócios não varejistas por cidade
*   **CHAS** - variável dummy do Rio Charles (1 se a área faz divisa com o rio; 0 caso contrário)
*   **NOX** - concentração de óxidos nítricos (partes por 10 milhões)
*   **RM** - número médio de cômodos por habitação
*   **AGE** - proporção de unidades ocupadas pelos proprietários construídas antes de 1940
*   **DIS** - distâncias ponderadas para cinco centros de empregos de Boston
*   **RAD** - índice de acessibilidade a rodovias radiais
*   **TAX** - taxa de imposto sobre a propriedade de valor total por $10.000
*   **PTRATIO** - razão aluno-professor por cidade
*   **B** - 1000(Bk - 0,63)^2, onde Bk é a proporção de negros por cidade
*   **LSTAT** - % de status inferior da população
*   **MEDV** - Valor mediano das casas ocupadas pelos proprietários em milhares de dólares

In [ ]:
import numpy as np
import pandas as pd
from pandas import read_csv
column_names = ['CRIM', 'ZN', 'INDUS', 'CHAS', 'NOX', 'RM', 'AGE', 'DIS', 'RAD', 'TAX', 'PTRATIO', 'B', 'LSTAT', 'MEDV']
data = read_csv('./housing.csv', header=None, delimiter=r"\s+", names=column_names)
print(data.head(5))

      CRIM    ZN  INDUS  CHAS    NOX     RM   AGE     DIS  RAD    TAX  \
0  0.00632  18.0   2.31     0  0.538  6.575  65.2  4.0900    1  296.0   
1  0.02731   0.0   7.07     0  0.469  6.421  78.9  4.9671    2  242.0   
2  0.02729   0.0   7.07     0  0.469  7.185  61.1  4.9671    2  242.0   
3  0.03237   0.0   2.18     0  0.458  6.998  45.8  6.0622    3  222.0   
4  0.06905   0.0   2.18     0  0.458  7.147  54.2  6.0622    3  222.0   

   PTRATIO       B  LSTAT  MEDV  
0     15.3  396.90   4.98  24.0  
1     17.8  396.90   9.14  21.6  
2     17.8  392.83   4.03  34.7  
3     18.7  394.63   2.94  33.4  
4     18.7  396.90   5.33  36.2  


# **Exercício**

A fim  responder os items a seguir considere a seguintes seguintes **features ** CRIM, ZN , INDUS, CHAS, NOX , RM, AGE, DIS, RAD, TAX, PTRATIO,B, LSTAT, bem como a variável **target**  MEDV. Em seus estudos assuma uma proporção de 75% dos dados para treinamento e 25% do dados para teste.

Obs.Reveja os exercício  AirQuality e Airfoil ambos feitos em sala.

**Item 1** Realize a análise exploratória dos dados.

**Item 2** Construa a curva de aprendizado, relacionando a função de perda (MSE) em função do número de épocas.

**Item 3** Construa um gráfico dos valores previstos e dos valores reais em torno da reta identidade.

**Item 4** Construa um gráfico contendo os valores previstos e os valores reais, de modo a comparar as duas curvas.

**Item 5** Construa uma estratégia de busca em grade (GridSearch) a fim de encontrar os melhores parâmetros para a rede neural.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.neural_network import MLPRegressor
from sklearn.metrics import mean_squared_error, r2_score

sns.set(style='whitegrid')
RANDOM_STATE = 42


In [ ]:
# Separacao de features e target
X = data.drop(columns=['MEDV'])
y = data['MEDV']

# Split 75% treino e 25% teste
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=RANDOM_STATE
)

print(f'Tamanho treino: {X_train.shape[0]} amostras')
print(f'Tamanho teste: {X_test.shape[0]} amostras')


## Item 1 - Analise exploratoria dos dados


In [ ]:
print('Informacoes gerais:')
display(data.info())

print('\nEstatisticas descritivas:')
display(data.describe())

print('\nValores ausentes por coluna:')
display(data.isnull().sum())


In [ ]:

data.hist(figsize=(16, 12), bins=20)
plt.suptitle('Distribuicao das variaveis', y=1.02)
plt.tight_layout()
plt.show()


In [ ]:

plt.figure(figsize=(12, 9))
corr = data.corr(numeric_only=True)
sns.heatmap(corr, cmap='coolwarm', center=0)
plt.title('Matriz de correlacao')
plt.show()

print('Correlacao com MEDV (ordenada):')
display(corr['MEDV'].sort_values(ascending=False))


## Item 2 - Curva de aprendizado (MSE x epocas)


In [ ]:
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

epochs = 200
mlp_epoch = MLPRegressor(
    hidden_layer_sizes=(64, 32),
    activation='relu',
    solver='adam',
    alpha=0.0001,
    learning_rate_init=0.001,
    max_iter=1,
    warm_start=True,
    random_state=RANDOM_STATE
)

train_mse, test_mse = [], []

for _ in range(epochs):
    mlp_epoch.fit(X_train_scaled, y_train)
    pred_train = mlp_epoch.predict(X_train_scaled)
    pred_test = mlp_epoch.predict(X_test_scaled)
    train_mse.append(mean_squared_error(y_train, pred_train))
    test_mse.append(mean_squared_error(y_test, pred_test))

plt.figure(figsize=(10, 5))
plt.plot(range(1, epochs + 1), train_mse, label='MSE Treino')
plt.plot(range(1, epochs + 1), test_mse, label='MSE Teste')
plt.xlabel('Epocas')
plt.ylabel('MSE')
plt.title('Curva de aprendizado - Rede Neural')
plt.legend()
plt.show()


## Itens 3 e 4 - Comparacao entre valores reais e previstos


In [ ]:

model_base = Pipeline([
    ('scaler', StandardScaler()),
    ('mlp', MLPRegressor(
        hidden_layer_sizes=(64, 32),
        random_state=RANDOM_STATE,
        max_iter=2000
    ))
])

model_base.fit(X_train, y_train)
y_pred = model_base.predict(X_test)

mse_base = mean_squared_error(y_test, y_pred)
r2_base = r2_score(y_test, y_pred)
print(f'MSE (teste): {mse_base:.4f}')
print(f'R2  (teste): {r2_base:.4f}')


In [ ]:

plt.figure(figsize=(7, 7))
plt.scatter(y_test, y_pred, alpha=0.7)
min_val = min(y_test.min(), y_pred.min())
max_val = max(y_test.max(), y_pred.max())
plt.plot([min_val, max_val], [min_val, max_val], 'r--', linewidth=2)
plt.xlabel('Valores reais (MEDV)')
plt.ylabel('Valores previstos (MEDV)')
plt.title('Previstos vs Reais com reta identidade')
plt.show()


In [ ]:

comp = pd.DataFrame({'Real': y_test.values, 'Previsto': y_pred})
comp = comp.sort_values('Real').reset_index(drop=True)

plt.figure(figsize=(12, 5))
plt.plot(comp['Real'].values, label='Real', linewidth=2)
plt.plot(comp['Previsto'].values, label='Previsto', linewidth=2)
plt.xlabel('Amostras (ordenadas por valor real)')
plt.ylabel('MEDV')
plt.title('Comparacao entre curvas: real x previsto')
plt.legend()
plt.show()


## Item 5 - GridSearch para melhores hiperparametros


In [ ]:
pipe = Pipeline([
    ('scaler', StandardScaler()),
    ('mlp', MLPRegressor(random_state=RANDOM_STATE, max_iter=2000))
])

param_grid = {
    'mlp__hidden_layer_sizes': [(32,), (64,), (64, 32), (128, 64)],
    'mlp__activation': ['relu', 'tanh'],
    'mlp__alpha': [0.0001, 0.001, 0.01],
    'mlp__learning_rate_init': [0.001, 0.01]
}

grid = GridSearchCV(
    estimator=pipe,
    param_grid=param_grid,
    scoring='neg_mean_squared_error',
    cv=5,
    n_jobs=-1,
    verbose=0
)

grid.fit(X_train, y_train)

best_model = grid.best_estimator_
best_pred = best_model.predict(X_test)
best_mse = mean_squared_error(y_test, best_pred)
best_r2 = r2_score(y_test, best_pred)

print('Melhores hiperparametros encontrados:')
print(grid.best_params_)
print(f'\nMelhor score CV (neg MSE): {grid.best_score_:.4f}')
print(f'MSE no teste (modelo otimizado): {best_mse:.4f}')
print(f'R2 no teste (modelo otimizado): {best_r2:.4f}')
